# 11 — WELFake Exploratory Data Analysis

**Mục tiêu:** Phân tích phân phối, độ dài, từ vựng của WELFake. So sánh với ISOT để hiểu domain shift.

**Input:** `data/processed/preprocessed_welfake_full.csv`  
**Output:** Biểu đồ trong `reports/welfake_*.png`

## 1. Import & Load

In [8]:
import warnings
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')
ROOT    = Path('..').resolve()
REPORTS = ROOT / 'reports'
REPORTS.mkdir(exist_ok=True)

# Cố gắng import WordCloud (optional)
try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False
    print('wordcloud not installed — skipping word cloud plots')

df = pd.read_csv(ROOT / 'data' / 'processed' / 'preprocessed_welfake_full.csv')
df['label'] = df['label'].astype(int)
print(f'WELFake loaded: {df.shape}')
print(df['label'].value_counts().rename({0: 'REAL (0)', 1: 'FAKE (1)'}))

WELFake loaded: (72074, 2)
label
FAKE (1)    37046
REAL (0)    35028
Name: count, dtype: int64


## 2. Phân phối Nhãn

In [9]:
vc = df['label'].value_counts().sort_index()
labels_name = ['REAL (0)', 'FAKE (1)']
colors = ['#2ecc71', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar
axes[0].bar(labels_name, vc.values, color=colors, edgecolor='white', linewidth=1.5)
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Phân phối nhãn — WELFake', fontsize=12)
axes[0].set_ylabel('Số bài', fontsize=11)
axes[0].set_ylim(0, max(vc.values) * 1.15)
axes[0].grid(axis='y', alpha=0.3)

# Pie
axes[1].pie(vc.values, labels=labels_name, colors=colors, autopct='%1.1f%%',
            startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Tỉ lệ nhãn — WELFake', fontsize=12)

plt.tight_layout()
plt.savefig(REPORTS / 'welfake_label_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Imbalance ratio: {max(vc)/min(vc):.3f}  (1.0 = perfectly balanced)')
print('Saved → reports/welfake_label_dist.png')

Imbalance ratio: 1.058  (1.0 = perfectly balanced)
Saved → reports/welfake_label_dist.png


## 3. Phân phối Độ dài Văn bản

In [10]:
df['word_count'] = df['processed_text'].str.split().str.len()

real_wc = df.loc[df['label']==0, 'word_count']
fake_wc = df.loc[df['label']==1, 'word_count']

print('Độ dài văn bản (số từ sau preprocessing):')
print(f'  FAKE — mean={fake_wc.mean():.0f}, median={fake_wc.median():.0f}, std={fake_wc.std():.0f}')
print(f'  REAL — mean={real_wc.mean():.0f}, median={real_wc.median():.0f}, std={real_wc.std():.0f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
cap = df['word_count'].quantile(0.99)

for ax, data, label, color in zip(axes,
    [fake_wc, real_wc], ['FAKE', 'REAL'], ['#e74c3c', '#2ecc71']):
    ax.hist(data.clip(upper=cap), bins=60, color=color, alpha=0.75, edgecolor='white')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1.5, label=f'Median={data.median():.0f}')
    ax.axvline(data.mean(), color='blue', linestyle=':', linewidth=1.5, label=f'Mean={data.mean():.0f}')
    ax.set_title(f'{label} — Phân phối độ dài', fontsize=11)
    ax.set_xlabel('Số từ (sau preprocessing, cắt tại p99)', fontsize=10)
    ax.set_ylabel('Số bài', fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle('WELFake — Độ dài văn bản theo nhãn', fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS / 'welfake_text_length_by_label.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/welfake_text_length_by_label.png')

Độ dài văn bản (số từ sau preprocessing):
  FAKE — mean=283, median=209, std=358
  REAL — mean=330, median=255, std=300
Saved → reports/welfake_text_length_by_label.png


## 4. Word Cloud

In [11]:
if HAS_WORDCLOUD:
    for label, name, color_map in [(1, 'FAKE', 'Reds'), (0, 'REAL', 'Greens')]:
        corpus = ' '.join(df.loc[df['label']==label, 'processed_text'].dropna())
        wc = WordCloud(
            width=900, height=450, background_color='white',
            max_words=150, colormap=color_map, collocations=False
        ).generate(corpus)
        fig, ax = plt.subplots(figsize=(11, 5))
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(f'WELFake — Word Cloud ({name})', fontsize=13)
        plt.tight_layout()
        fname = REPORTS / f'welfake_wordcloud_{name.lower()}.png'
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved → {fname}')
else:
    print('Bỏ qua word cloud (wordcloud không được cài). Chạy: pip install wordcloud')

Saved → D:\PROJECT_GIT\Fake-News-Detection\reports\welfake_wordcloud_fake.png
Saved → D:\PROJECT_GIT\Fake-News-Detection\reports\welfake_wordcloud_real.png


## 5. Top 20 Unigrams & Bigrams

In [12]:
def plot_top_ngrams(texts, label_name, color, ax_uni, ax_bi, top_n=20):
    # Unigrams
    cv1 = CountVectorizer(max_features=top_n, ngram_range=(1,1))
    cv1.fit_transform(texts)
    uni_freq = dict(zip(cv1.get_feature_names_out(),
                        cv1.transform(texts).sum(axis=0).A1))
    uni_sorted = sorted(uni_freq.items(), key=lambda x: x[1], reverse=True)[:top_n]
    words1, counts1 = zip(*uni_sorted)
    ax_uni.barh(range(top_n), counts1, color=color, alpha=0.75)
    ax_uni.set_yticks(range(top_n))
    ax_uni.set_yticklabels(words1, fontsize=9)
    ax_uni.invert_yaxis()
    ax_uni.set_title(f'{label_name} — Top {top_n} Unigrams', fontsize=11)
    ax_uni.set_xlabel('Tần suất', fontsize=9)

    # Bigrams
    cv2 = CountVectorizer(max_features=top_n, ngram_range=(2,2))
    cv2.fit_transform(texts)
    bi_freq = dict(zip(cv2.get_feature_names_out(),
                       cv2.transform(texts).sum(axis=0).A1))
    bi_sorted = sorted(bi_freq.items(), key=lambda x: x[1], reverse=True)[:top_n]
    words2, counts2 = zip(*bi_sorted)
    ax_bi.barh(range(top_n), counts2, color=color, alpha=0.75)
    ax_bi.set_yticks(range(top_n))
    ax_bi.set_yticklabels(words2, fontsize=9)
    ax_bi.invert_yaxis()
    ax_bi.set_title(f'{label_name} — Top {top_n} Bigrams', fontsize=11)
    ax_bi.set_xlabel('Tần suất', fontsize=9)

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fake_texts = df.loc[df['label']==1, 'processed_text'].fillna('').tolist()
real_texts = df.loc[df['label']==0, 'processed_text'].fillna('').tolist()

plot_top_ngrams(fake_texts, 'FAKE', '#e74c3c', axes[0][0], axes[0][1])
plot_top_ngrams(real_texts, 'REAL', '#2ecc71', axes[1][0], axes[1][1])

plt.suptitle('WELFake — Top 20 N-grams per Class', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(REPORTS / 'welfake_top_ngrams.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/welfake_top_ngrams.png')

Saved → reports/welfake_top_ngrams.png


## 6. So sánh WELFake vs ISOT

In [14]:
# Cố tải ISOT để so sánh
isot_path = ROOT / 'data' / 'processed' / 'preprocessed_isot_full.csv'
if isot_path.exists():
    isot = pd.read_csv(isot_path)
    isot['word_count'] = isot['processed_text'].str.split().str.len()

    comparison = pd.DataFrame({
        'Metric': [
            'Tổng mẫu', 'FAKE', 'REAL', 'Tỉ lệ FAKE%',
            'Mean words', 'Median words', 'Vocabulary (~unique tokens)'
        ],
        'ISOT': [
            len(isot),
            (isot['label']==1).sum(),
            (isot['label']==0).sum(),
            f"{(isot['label']==1).mean()*100:.1f}%",
            f"{isot['word_count'].mean():.0f}",
            f"{isot['word_count'].median():.0f}",
            f"{isot['processed_text'].str.split().explode().nunique():,}"
        ],
        'WELFake': [
            len(df),
            (df['label']==1).sum(),
            (df['label']==0).sum(),
            f"{(df['label']==1).mean()*100:.1f}%",
            f"{df['word_count'].mean():.0f}",
            f"{df['word_count'].median():.0f}",
            f"{df['processed_text'].str.split().explode().nunique():,}"
        ]
    })
    print(comparison.to_string(index=False))

    # Bar chart so sánh
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    datasets = ['ISOT', 'WELFake']
    totals = [len(isot), len(df)]
    fake_counts = [(isot['label']==1).sum(), (df['label']==1).sum()]
    real_counts = [(isot['label']==0).sum(), (df['label']==0).sum()]

    x = range(2)
    axes[0].bar(x, fake_counts, label='FAKE', color='#e74c3c', alpha=0.8)
    axes[0].bar(x, real_counts, bottom=fake_counts, label='REAL', color='#2ecc71', alpha=0.8)
    axes[0].set_xticks(x); axes[0].set_xticklabels(datasets, fontsize=11)
    axes[0].set_title('Số mẫu theo dataset', fontsize=11)
    axes[0].set_ylabel('Số bài', fontsize=10)
    axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

    axes[1].boxplot(
        [isot['word_count'].clip(upper=500), df['word_count'].clip(upper=500)],
        labels=datasets, patch_artist=True,
        boxprops=dict(facecolor='steelblue', alpha=0.6)
    )
    axes[1].set_title('Phân phối độ dài (clip tại 500)', fontsize=11)
    axes[1].set_ylabel('Số từ', fontsize=10)
    axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle('WELFake vs ISOT — Tổng quan', fontsize=12)
    plt.tight_layout()
    plt.savefig(REPORTS / 'welfake_isot_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → reports/welfake_isot_comparison.png')
else:
    print('ISOT processed file không tìm thấy — bỏ qua so sánh')

                     Metric    ISOT WELFake
                   Tổng mẫu   38653   72074
                       FAKE   17457   37046
                       REAL   21196   35028
                Tỉ lệ FAKE%   45.2%   51.4%
                 Mean words     235     306
               Median words     213     230
Vocabulary (~unique tokens) 194,086 294,239
Saved → reports/welfake_isot_comparison.png


## Tóm tắt EDA

| Quan sát | Ý nghĩa cho Phase 8 |
|----------|---------------------|
| WELFake gần balanced hơn ISOT | Model trained trên ISOT có thể bias khi test trên WELFake |
| Độ dài trung bình WELFake < ISOT | ISOT (Reuters) dài và formal hơn — domain shift về style |
| Từ vựng WELFake đa dạng hơn | TF-IDF 5000 features có thể không capture đủ WELFake patterns |